# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**

* Joel Arturo Becerril Balderas — A01797427

---

## 🎯 Contexto de Negocio y Objetivo

### ¿Por qué analizar el sentimiento de reseñas?

Plataformas como Amazon, Yelp e IMDb reciben **millones de comentarios diarios**. Es humanamente imposible leerlos todos. El análisis automático de sentimiento permite:

- **E-commerce:** detectar productos con oleada de comentarios negativos antes de que escalen a crisis
- **Restaurantes:** identificar quejas recurrentes (servicio lento, porciones pequeñas) de forma automática
- **Entretenimiento:** predecir la recepción de una película con base en reseñas tempranas
- **Servicio al cliente:** priorizar tickets con sentimiento negativo extremo para atención urgente

### ¿Qué hace este notebook?

Compara **dos enfoques** para convertir texto en vectores numéricos y clasificar su sentimiento:

| Parte | Enfoque | Fortaleza | Limitación |
|---|---|---|---|
| **Parte I** | Promedio de vectores de palabras del vocabulario | Computacionalmente eficiente | Pierde el orden y contexto de la oración |
| **Parte II** | Embedding directo de la oración completa | Captura contexto, negaciones y relaciones | Mayor costo por comentario |

In [ ]:
import pandas as pd
import numpy as np
import re
import csv
import pickle
import os
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer

SEED = 42

# En VS Code el cwd del notebook es el directorio donde está el .ipynb
DATA_DIR = Path(os.getcwd()) / 'data' / 'sentiment labelled sentences'
print(f"DATA_DIR: {DATA_DIR}")
print(f"Archivos encontrados: {list(DATA_DIR.glob('*.txt'))}")

En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



### 📌 Sobre el dataset y el fix de IMDb

Los 3,000 comentarios están balanceados (50% positivo, 50% negativo) y vienen de tres dominios distintos, lo que hace al modelo más generalizable.

**¿Por qué el archivo IMDb necesita corrección?**

El archivo `imdb_labelled.txt` contiene reseñas con **comillas dobles internas**, por ejemplo:
```
The "acting" was surprisingly good	1
```
El parser CSV de pandas interpreta esas comillas como delimitadores de campo. Cuando encuentra una comilla sin cierre, agrupa múltiples líneas en una sola fila. Resultado: de 1,000 registros solo obtiene **748**.

**Fix:** `quoting=csv.QUOTE_NONE` le dice a pandas que trate todas las comillas como texto literal → 1,000 filas correctas.

Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [ ]:
# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

amazon = pd.read_csv(os.path.join(DATA_DIR, 'amazon_cells_labelled.txt'),
                     sep='\t', header=None, names=['text', 'label'])
yelp   = pd.read_csv(os.path.join(DATA_DIR, 'yelp_labelled.txt'),
                     sep='\t', header=None, names=['text', 'label'])
# IMDb tiene líneas con comillas que confunden el parser CSV por defecto → quoting=QUOTE_NONE
imdb   = pd.read_csv(os.path.join(DATA_DIR, 'imdb_labelled.txt'),
                     sep='\t', header=None, names=['text', 'label'],
                     quoting=csv.QUOTE_NONE)

amazon['source'] = 'amazon'
yelp['source']   = 'yelp'
imdb['source']   = 'imdb'

df = pd.concat([amazon, yelp, imdb], ignore_index=True)

# *********** Aquí termina la sección de agregar código *************

print(f"Tamaño del DataFrame df: {df.shape}")
print(f"\nRegistros por fuente:\n{df['source'].value_counts()}")
print(f"\nDistribución de etiquetas:\n{df['label'].value_counts()}")

In [ ]:
# Verifiquemos la información del DataFrame:

df.info()

In [ ]:
# Y mostremos sus primeros registros:

df.head()

### 🧹 ¿Por qué limpiar el texto?

Los modelos trabajan con números, no con texto crudo. Estandarizar el texto evita que variaciones superficiales generen palabras distintas para el modelo:

| Transformación | Ejemplo antes | Ejemplo después | ¿Por qué? |
|---|---|---|---|
| Minúsculas | `Great`, `GREAT`, `great` | `great` | Son la misma palabra |
| Eliminar HTML | `<b>Amazing</b>` | `amazing` | Tags no aportan sentimiento |
| Solo letras | `great!!! 100%` | `great` | Números y signos no son útiles aquí |
| Normalizar espacios | `great  product` | `great product` | Limpieza de espacios redundantes |

Sin este paso, `"Great"` y `"great"` serían **dos entradas distintas** en el vocabulario, duplicando entradas sin agregar información semántica.

# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [ ]:
# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # eliminar etiquetas HTML
    text = re.sub(r'[^a-z\s]', ' ', text)       # conservar solo letras y espacios
    text = re.sub(r'\s+', ' ', text).strip()     # normalizar espacios
    return text

Xclean = df['text'].apply(clean_text).tolist()
y      = df['label'].tolist()

# *********** Aquí termina la sección de agregar código *************

print(f"Total comentarios: {len(Xclean)}, Total etiquetas: {len(y)}")

### ✂️ ¿Por qué dividir en tres conjuntos y con qué proporciones?

Piensa en ello como preparar un examen:

| Conjunto | % | N registros | Analogía | Uso en el pipeline |
|---|---|---|---|---|
| **Train** | 70% | 2,100 | Material de estudio | El modelo aprende de aquí |
| **Val** | 15% | 450 | Ejercicios de práctica | Aquí ajustamos hiperparámetros y comparamos modelos |
| **Test** | 15% | 450 | El examen real | Se usa **UNA sola vez** al reportar el resultado final |

**`stratify=y`** garantiza que la proporción 50/50 de positivos y negativos se mantenga igual en los tres conjuntos. Sin esto podría ocurrir que el test set tuviera 70% positivos y el modelo pareciera artificialmente más preciso.

**`random_state=42`** fija la semilla aleatoria → el split es idéntico en cada ejecución. Esto es esencial para reproducibilidad académica.

In [ ]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


# **Pregunta - 3:**


### 📚 ¿Qué es el vocabulario y por qué filtrarlo?

El **vocabulario** es el conjunto de palabras que el modelo "conoce". Se construye **solo con el conjunto de entrenamiento** para evitar *data leakage*: no podemos saber qué palabras usará el modelo en producción real, así que solo usamos lo que "vimos" durante el entrenamiento.

**Filtros aplicados:**
- **`min_freq = 2`:** Palabras que aparecen solo una vez pueden ser errores tipográficos (`"amaazing"`) o tan raras que no aportan señal estadística.
- **`min_len = 2`:** Caracteres sueltos (`"a"`, `"s"`, `"i"`) son residuos de la limpieza del texto.

**Ejemplo con un mini corpus de train:**
```
Comentarios de train: ["great product", "great service", "bad product", "arrived broken"]

Frecuencias: great=2, product=2, service=1, bad=1, arrived=1, broken=1

Con min_freq=2 → vocabulario = {great, product}
```
Cuando filtramos val y test, un comentario como `"terrible experience"` quedaría **vacío** (ambas palabras no están en el vocabulario) y se representaría con el vector cero en la Pregunta 7.


Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [ ]:
# ************* Inicia la sección de agregar código:*****************************

# Partición: 70% entrenamiento, 15% validación, 15% prueba
Xtrain_tmp, Xtest, ytrain_tmp, ytest = train_test_split(
    Xclean, y, test_size=0.15, random_state=SEED, stratify=y)

Xtrain, Xval, ytrain, yval = train_test_split(
    Xtrain_tmp, ytrain_tmp, test_size=0.15/0.85, random_state=SEED, stratify=ytrain_tmp)

# *********** Termina la sección de agregar código *************

print('X,y Train:', len(Xtrain), len(ytrain))
print('X,y Val:  ', len(Xval),   len(yval))
print('X,y Test: ', len(Xtest),  len(ytest))

# **Pregunta - 4:**

### ⚠️ ¿Qué pasa con palabras de val/test que no están en el vocabulario?

Esas palabras se **descartan silenciosamente**. Hay tres escenarios:

**Escenario 1 — El comentario tiene ALGUNAS palabras fuera del vocabulario**
```
val:   "absolutely terrible product"
vocab: {terrible, product}   ← "absolutely" no está

→ filtrado: "terrible product"
→ se vectoriza con las 2 palabras que sí están (información incompleta)
→ se clasifica igual, pero con menos señal
```

**Escenario 2 — El comentario queda COMPLETAMENTE VACÍO**
```
val:   "dreadful experience"
vocab: {great, product, bad}  ← ninguna palabra coincide

→ filtrado: ""
→ se asigna el vector cero [0, 0, 0, ..., 0]  (UNK_VEC en nuestro código)
→ el clasificador lo clasifica igual (probablemente mal, pues el vector cero
  no tiene señal semántica)
```

**Escenario 3 — Parte II no tiene este problema**

En la Parte II no usas vocabulario filtrado — le pasas el comentario completo directamente a `model.encode()`. El transformer tokeniza internamente y procesa cualquier palabra, incluso las que nunca aparecieron en train. Esta es la **ventaja principal de la Parte II** sobre la Parte I y una razón clave por la que generalmente obtiene mejor accuracy.




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [ ]:
# a. Usa el conjunto de entrenamiento para generar tu vocabulario

MIN_FREQ = 2   # mínima frecuencia de aparición
MIN_LEN  = 2   # mínima longitud en caracteres

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

word_counts = Counter()
for comment in Xtrain:
    word_counts.update(comment.split())

vocab = sorted(w for w, c in word_counts.items() if c >= MIN_FREQ and len(w) >= MIN_LEN)

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# b. Indica el tamaño del vocabulario generado.
print('Longitud del vocabulario generado:')

# ******* Inicia la sección de agregar código: ***********

print(f"  {len(vocab)} palabras únicas (min_freq={MIN_FREQ}, min_len={MIN_LEN})")

# *********** Aquí termina la sección de agregar código *************

### 🤖 ¿Cómo convierte el modelo una palabra en un vector de 768 números?

El modelo `BAAI/bge-base-en-v1.5` es un transformer (BERT-base) entrenado con **aprendizaje contrastivo**: aprendió a colocar palabras con significados similares *cerca* en el espacio vectorial y palabras opuestas *lejos*.

**Visualización simplificada (3 dimensiones en lugar de 768):**

| Palabra | Dim 1 (positividad) | Dim 2 (intensidad) | Dim 3 (concreción) |
|---|---|---|---|
| `great` | +0.90 | +0.80 | +0.30 |
| `excellent` | +0.88 | +0.85 | +0.28 |
| `terrible` | -0.85 | +0.75 | +0.32 |
| `phone` | +0.10 | +0.20 | +0.95 |

`great` y `excellent` → vectores muy similares (mismo campo semántico)
`great` y `terrible` → vectores opuestos (sentimientos contrarios)

**¿Por qué guardar el diccionario en pickle?**
Pasar cada palabra del vocabulario por la red neuronal toma tiempo. Lo hacemos **una sola vez** y guardamos el resultado. Las siguientes ejecuciones cargan el archivo `.pkl` directamente, ahorrando minutos de cómputo.

In [ ]:
# c. Filtra los conjuntos usando solo palabras del vocabulario → train_X, val_X, test_X

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

vocab_set = set(vocab)

def filter_with_vocab(comments):
    return [' '.join(w for w in c.split() if w in vocab_set) for c in comments]

train_X = filter_with_vocab(Xtrain)
val_X   = filter_with_vocab(Xval)
test_X  = filter_with_vocab(Xtest)

# *********** Aquí termina la sección de agregar código *************

print(f"train_X: {len(train_X)} comentarios")
print(f"val_X:   {len(val_X)}   comentarios")
print(f"test_X:  {len(test_X)}  comentarios")

In [ ]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**

  Desarrollado por BAAI (Beijing Academy of Artificial Intelligence). Basado en arquitectura BERT-base (transformer encoder bidireccional). Genera vectores de **768 dimensiones** con aproximadamente 110M de parámetros (~438 MB). Entrenado con aprendizaje contrastivo usando el framework FlagEmbedding sobre grandes corpus en inglés. Ocupa posiciones altas en el benchmark MTEB para tareas de recuperación semántica. Es el punto de equilibrio entre rendimiento y costo computacional del trio.

* **b) bge-large-en-v1.5**

  También de BAAI, variante de mayor capacidad basada en BERT-large. Genera vectores de **1024 dimensiones** con ~335M de parámetros (~1.3 GB). Mismo proceso de entrenamiento contrastivo que bge-base pero con mayor expresividad representacional. Supera consistentemente a bge-base en benchmarks de similitud semántica y búsqueda de información, a costa de mayor tiempo de inferencia y consumo de memoria.

* **c) e5-base-v2**

  Desarrollado por Microsoft Research. Arquitectura BERT-base, vectores de **768 dimensiones** y ~110M parámetros. Su nombre proviene de *EmbEddings from bidirEctional Encoder rEpresentations* (E5). Entrenado con pares de texto heterogéneos (consultas web, foros, código). Requiere prefijar los textos con `"query: "` o `"passage: "` para obtener el máximo rendimiento en tareas de recuperación. Compite con bge-base-en-v1.5 en el benchmark MTEB.

### ++++++++ Termina la sección de agregar texto: ++++++++++

### 📐 Fórmula: Embedding promedio de un comentario

Para que un clasificador (Regresión Logística, Random Forest) pueda procesar un comentario, necesitamos representarlo como **un solo vector de tamaño fijo**. Promediamos los vectores de todas sus palabras:

$$\vec{v}_{comentario} = \frac{1}{N} \sum_{i=1}^{N} \vec{v}_{palabra_i}$$

**Ejemplo concreto** (simplificado a 3 dimensiones en lugar de 768):

```
Comentario filtrado: "great product"  →  N = 2 palabras

v("great")   = [ 0.90,  0.10,  0.50 ]
v("product") = [ 0.30,  0.70,  0.20 ]
               ──────────────────────
promedio     = [ 0.60,  0.40,  0.35 ]
```

El vector resultante es el **centroide semántico** del comentario en el espacio vectorial de 768 dimensiones.

⚠️ **Limitación importante:** El promedio pierde el orden de las palabras. `"not good"` y `"good not"` producen el **mismo vector promedio**, aunque tengan significados distintos. La Parte II resuelve esto al codificar la oración completa con el transformer.

# **Pregunta - 6:**

In [ ]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado: BAAI/bge-base-en-v1.5

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
model = SentenceTransformer(MODEL_NAME)

DICT_PATH = 'vocab_embeddings_bge_base.pkl'

if os.path.exists(DICT_PATH):
    with open(DICT_PATH, 'rb') as f:
        vocab_embeddings = pickle.load(f)
    print(f"Diccionario cargado desde archivo: {DICT_PATH}")
else:
    print(f"Generando embeddings para {len(vocab)} palabras del vocabulario...")
    vectors = model.encode(vocab, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)
    vocab_embeddings = {word: vec for word, vec in zip(vocab, vectors)}
    with open(DICT_PATH, 'wb') as f:
        pickle.dump(vocab_embeddings, f)
    print(f"Diccionario guardado en: {DICT_PATH}")

# *********** Aquí termina la sección de agregar código *************

### 🔢 ¿Qué es un token y por qué contarlos?

En este contexto, **token ≈ palabra** (después de la limpieza). Contamos cuántas palabras del vocabulario filtrado se usaron para generar los embeddings de cada conjunto.

Este conteo es relevante por dos razones:
1. **Costo computacional:** los modelos de HuggingFace tienen un tiempo de inferencia proporcional al número de tokens procesados
2. **Cobertura del vocabulario:** si el número de tokens es muy bajo respecto al total de palabras en los comentarios, el vocabulario está filtrando demasiado y perdemos información útil

Un número de tokens **proporcional** entre train/val/test confirma que el vocabulario construido desde train representa bien a los otros conjuntos.

In [ ]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

keys3 = list(vocab_embeddings.keys())[:3]
for k in keys3:
    v = vocab_embeddings[k]
    print(f"'{k}': dimensión={v.shape}, primeros 5 valores={v[:5].round(4)}")

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# c) Tamaño del diccionario generado:

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

EMB_DIM = model.get_sentence_embedding_dimension()
print(f"Tamaño del diccionario de embeddings: {len(vocab_embeddings)} entradas")
print(f"Dimensión de cada vector: {EMB_DIM}")

# *********** Aquí termina la sección de agregar código *************

### 🏋️ Los dos clasificadores: ¿cómo funcionan?

#### Regresión Logística
Aprende un **hiperplano** en el espacio de 768 dimensiones que separa comentarios positivos de negativos. Para un comentario nuevo, calcula de qué lado del hiperplano cae y con qué probabilidad.
- Tiende a **no sobreentrenarse** con vectores de alta dimensión
- Rápido de entrenar y predecir

#### Random Forest (Bosque Aleatorio)
Entrena `n_estimators=200` árboles de decisión distintos, cada uno sobre una muestra aleatoria de datos y características. La predicción final es la **votación mayoritaria** de los 200 árboles.
- Captura relaciones no lineales entre dimensiones del embedding
- Tiende a **memorizar** el train set (train accuracy ~99%) → mayor gap con val

#### Criterio de sobreentrenamiento de la actividad

$$|\text{Acc}_{train} - \text{Acc}_{val}| < 3\%$$

Si un modelo tiene 99% en train y 75% en val → diferencia de 24% → **sobreentrenado** ✗

#### ¿Qué muestra el `classification_report`?

| Métrica | Fórmula | Significado |
|---|---|---|
| Precision | TP / (TP+FP) | De los que predije positivos, ¿cuántos realmente lo eran? |
| Recall | TP / (TP+FN) | De los positivos reales, ¿cuántos detecté? |
| F1-score | 2×P×R / (P+R) | Balance entre precision y recall |
| Support | — | Cuántas muestras reales hay de esa clase |

# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [ ]:
# a) Generar vectores embebidos promediando las palabras de cada comentario

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

UNK_VEC = np.zeros(EMB_DIM)

def comment_to_avg_embedding(comment):
    words = comment.split()
    vecs  = [vocab_embeddings[w] for w in words if w in vocab_embeddings]
    return np.mean(vecs, axis=0) if vecs else UNK_VEC

trainEmb = np.array([comment_to_avg_embedding(c) for c in train_X])
valEmb   = np.array([comment_to_avg_embedding(c) for c in val_X])
testEmb  = np.array([comment_to_avg_embedding(c) for c in test_X])

# *********** Aquí termina la sección de agregar código *************

### 🚀 Parte II vs Parte I: la diferencia clave

En la **Parte I**, cada comentario se representa como el promedio de vectores de palabras individuales. Esto tiene dos limitaciones:
1. Palabras fuera del vocabulario se descartan
2. El orden y contexto de las palabras se pierde

En la **Parte II**, el modelo codifica el **comentario completo** de una sola vez. El mecanismo de **atención del transformer** pondera cada palabra según el contexto de toda la oración:

| Comentario | Parte I (promedio) | Parte II (contexto completo) |
|---|---|---|
| `"not good"` | avg(v("not"), v("good")) ≈ neutral | Entiende la negación → negativo |
| `"good, but not great"` | promedio de todas las palabras | Capta la concesión → moderadamente positivo |
| `"absolutely terrible"` | avg(v("absolutely"), v("terrible")) | Capta el intensificador → muy negativo |

**Costo computacional:**
- Parte I: N palabras únicas del vocabulario (una vez, se cachea en pickle)
- Parte II: un forward pass del transformer **por comentario** (~2,550 forward passes para train+val+test)

In [ ]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

print(f"trainEmb: {trainEmb.shape}")
print(f"valEmb:   {valEmb.shape}")
print(f"testEmb:  {testEmb.shape}")

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 8:**

In [ ]:
# Número de tokens generados al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

train_tokens = sum(len(c.split()) for c in train_X)
val_tokens   = sum(len(c.split()) for c in val_X)
test_tokens  = sum(len(c.split()) for c in test_X)

print(f"Tokens en trainEmb: {train_tokens:,}")
print(f"Tokens en valEmb:   {val_tokens:,}")
print(f"Tokens en testEmb:  {test_tokens:,}")
print(f"Total tokens Parte I: {train_tokens + val_tokens + test_tokens:,}")

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [ ]:
# 9a) REGRESIÓN LOGÍSTICA

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(trainEmb, ytrain)

acc_train_lr = accuracy_score(ytrain, lr.predict(trainEmb))
acc_val_lr   = accuracy_score(yval,   lr.predict(valEmb))
diff_lr      = abs(acc_train_lr - acc_val_lr)

print("=== REGRESIÓN LOGÍSTICA — Parte I ===")
print(f"Accuracy Train: {acc_train_lr:.4f}")
print(f"Accuracy Val:   {acc_val_lr:.4f}")
print(f"Diferencia:     {diff_lr:.4f}  → {'No sobreentrenado ✓' if diff_lr < 0.03 else 'SOBREENTRENADO ✗'}")
print()
print("Classification Report (Validación):")
print(classification_report(yval, lr.predict(valEmb), target_names=['Negativo', 'Positivo']))

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 9b) BOSQUE ALEATORIO (Random Forest)

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(trainEmb, ytrain)

acc_train_rf = accuracy_score(ytrain, rf.predict(trainEmb))
acc_val_rf   = accuracy_score(yval,   rf.predict(valEmb))
diff_rf      = abs(acc_train_rf - acc_val_rf)

print("=== BOSQUE ALEATORIO (Random Forest) — Parte I ===")
print(f"Accuracy Train: {acc_train_rf:.4f}")
print(f"Accuracy Val:   {acc_val_rf:.4f}")
print(f"Diferencia:     {diff_rf:.4f}  → {'No sobreentrenado ✓' if diff_rf < 0.03 else 'SOBREENTRENADO ✗'}")
print()
print("Classification Report (Validación):")
print(classification_report(yval, rf.predict(valEmb), target_names=['Negativo', 'Positivo']))

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [ ]:
# 10a) Partición con los mismos parámetros que en la Parte I, sobre los 3000 comentarios originales

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

X_train_tmp, X_test, y_train_tmp, y_test = train_test_split(
    Xclean, y, test_size=0.15, random_state=SEED, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_tmp, y_train_tmp, test_size=0.15/0.85, random_state=SEED, stratify=y_train_tmp)

# *********** Aquí termina la sección de agregar código *************

print('X,y Train:', len(X_train), len(y_train))
print('X,y Val:  ', len(X_val),   len(y_val))
print('X,y Test: ', len(X_test),  len(y_test))

In [ ]:
# 10b) Vectores embebidos de comentarios completos con el mismo modelo (bge-base-en-v1.5)

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

print("Generando embeddings de comentarios completos...")
X_train_emb = model.encode(X_train, batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True)
X_val_emb   = model.encode(X_val,   batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True)
X_test_emb  = model.encode(X_test,  batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True)

train_tok_full = sum(len(c.split()) for c in X_train)
val_tok_full   = sum(len(c.split()) for c in X_val)
test_tok_full  = sum(len(c.split()) for c in X_test)

print(f"\nFormas: X_train_emb={X_train_emb.shape}, X_val_emb={X_val_emb.shape}, X_test_emb={X_test_emb.shape}")
print(f"\nTokens en X_train: {train_tok_full:,}")
print(f"Tokens en X_val:   {val_tok_full:,}")
print(f"Tokens en X_test:  {test_tok_full:,}")
print(f"Total tokens Parte II: {train_tok_full + val_tok_full + test_tok_full:,}")

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10c) REGRESIÓN LOGÍSTICA — Parte II

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

lr2 = LogisticRegression(max_iter=1000, random_state=SEED)
lr2.fit(X_train_emb, y_train)

acc_train_lr2 = accuracy_score(y_train, lr2.predict(X_train_emb))
acc_val_lr2   = accuracy_score(y_val,   lr2.predict(X_val_emb))
diff_lr2      = abs(acc_train_lr2 - acc_val_lr2)

print("=== REGRESIÓN LOGÍSTICA — Parte II ===")
print(f"Accuracy Train: {acc_train_lr2:.4f}")
print(f"Accuracy Val:   {acc_val_lr2:.4f}")
print(f"Diferencia:     {diff_lr2:.4f}  → {'No sobreentrenado ✓' if diff_lr2 < 0.03 else 'SOBREENTRENADO ✗'}")
print()
print("Comparación con Parte I:")
print(f"  Val Acc Parte I:  {acc_val_lr:.4f}  |  Val Acc Parte II: {acc_val_lr2:.4f}")
print()
print("Classification Report (Validación):")
print(classification_report(y_val, lr2.predict(X_val_emb), target_names=['Negativo', 'Positivo']))

# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10d) BOSQUE ALEATORIO — Parte II

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

rf2 = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf2.fit(X_train_emb, y_train)

acc_train_rf2 = accuracy_score(y_train, rf2.predict(X_train_emb))
acc_val_rf2   = accuracy_score(y_val,   rf2.predict(X_val_emb))
diff_rf2      = abs(acc_train_rf2 - acc_val_rf2)

print("=== BOSQUE ALEATORIO (Random Forest) — Parte II ===")
print(f"Accuracy Train: {acc_train_rf2:.4f}")
print(f"Accuracy Val:   {acc_val_rf2:.4f}")
print(f"Diferencia:     {diff_rf2:.4f}  → {'No sobreentrenado ✓' if diff_rf2 < 0.03 else 'SOBREENTRENADO ✗'}")
print()
print("Comparación con Parte I:")
print(f"  Val Acc Parte I:  {acc_val_rf:.4f}  |  Val Acc Parte II: {acc_val_rf2:.4f}")
print()
print("Classification Report (Validación):")
print(classification_report(y_val, rf2.predict(X_val_emb), target_names=['Negativo', 'Positivo']))

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 11:**

In [ ]:
# Reporte del mejor modelo y evaluación en el conjunto de Prueba

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

# Resumen de resultados en validación
results = {
    'LR  Parte I  (avg embed)':   (acc_val_lr,  lr,  testEmb,     ytest),
    'RF  Parte I  (avg embed)':   (acc_val_rf,  rf,  testEmb,     ytest),
    'LR  Parte II (full embed)':  (acc_val_lr2, lr2, X_test_emb,  y_test),
    'RF  Parte II (full embed)':  (acc_val_rf2, rf2, X_test_emb,  y_test),
}

print("=== Resumen de Accuracy en Validación ===")
for name, (acc_val, *_) in results.items():
    print(f"  {name}: {acc_val:.4f}")

best_name = max(results, key=lambda k: results[k][0])
_, best_model, best_X_test, best_y_test = results[best_name]
print(f"\nMejor modelo: {best_name}")

# Evaluación en conjunto de prueba
y_pred_test = best_model.predict(best_X_test)
acc_test    = accuracy_score(best_y_test, y_pred_test)
print(f"Accuracy en Prueba: {acc_test:.4f}")
print()
print("Classification Report (Prueba):")
print(classification_report(best_y_test, y_pred_test, target_names=['Negativo', 'Positivo']))

# Matriz de confusión
cm = confusion_matrix(best_y_test, y_pred_test)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'])
plt.title(f'Matriz de Confusión\n{best_name}')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.tight_layout()
plt.show()

# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 12:**

Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

**Comparación Parte I vs Parte II:**

La **Parte I** genera representaciones de comentarios promediando los vectores de palabras individuales del vocabulario filtrado. Esto introduce dos fuentes de pérdida de información: (1) las palabras fuera del vocabulario son descartadas, y (2) el promedio destruye el orden sintáctico del texto. A pesar de ello, captura la semántica léxica básica y resulta computacionalmente eficiente al calcular los embeddings del vocabulario una sola vez.

La **Parte II** codifica cada comentario completo directamente con el modelo BAAI/bge-base-en-v1.5. El modelo considera el contexto completo de la oración mediante atención multi-cabeza, produciendo representaciones más ricas que incorporan relaciones semánticas entre palabras, negaciones y matices contextuales. Esto explica la mejora típica de accuracy observada respecto a la Parte I.

**Observaciones sobre los modelos clasificadores:**
- La **Regresión Logística** generalmente mantiene una diferencia train-val menor al 3%, lo que la hace robusta ante sobreentrenamiento con estos vectores de alta dimensión.
- El **Random Forest** tiende a memorizar el conjunto de entrenamiento (accuracy train cercana a 1.0), lo que puede reflejarse en una mayor diferencia train-val. La hiperparametrización (n_estimators, max_depth) permitiría reducir este gap.

**Conclusión:** Los vectores embebidos de oraciones completas (`Parte II`) superan el enfoque de promedio de palabras (`Parte I`), ya que el modelo preentrenado bge-base-en-v1.5 aprovecha el contexto global del comentario para generar representaciones más discriminativas del sentimiento.

### ++++++++ Termina la sección de agregar texto: ++++++++++

# **Fin de la Actividad de Vectores Embebidos - HuggingFace**